# 02 - Capa Silver\n\nLimpieza, tipificación y normalización de datos.\n\nEn esta capa se transforma la fecha original `fecha_transaccion`, que viene con `/`, creando `fecha_venta` en formato estándar `yyyy-MM-dd`.

# Configuración base

Este notebook está preparado para ejecutarse en Google Colab conectado directamente a **Google Cloud Storage (GCS)** mediante autenticación nativa, reemplazando el uso de almacenamiento local o Google Drive.

Estructura esperada en el Bucket (`gs://data-lake-retail/`):

```text
data-lake-retail/
├── raw/
│   ├── venta_tiendas.csv
│   ├── Maestro_Producto.csv
│   ├── Maestro_Tienda.csv
│   └── venta_ecom.csv
├── bronze/
│   └── venta_tiendas_delta/      <-- (Origen: Datos leídos en este notebook)
├── silver/
│   └── venta_tiendas_delta/      <-- (Destino: Datos limpios guardados aquí)
├── gold/
└── evidencias/


In [1]:
# 1. Autenticación con Google Cloud
from google.colab import auth
auth.authenticate_user()
print("Autenticación con GCP exitosa.")

# 2. Instalación de dependencias para Colab
!pip uninstall -y dataproc-spark-connect opentelemetry-api importlib-metadata pyspark delta-spark > /dev/null
!pip install -q importlib-metadata==8.0.0 pyspark==3.4.1 delta-spark==2.4.0

# 3. Descargar el conector GCS manualmente a la carpeta de PySpark
import pyspark
import os
pyspark_jars_dir = os.path.join(pyspark.__path__[0], "jars")
!wget -q https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-2.2.14.jar -P {pyspark_jars_dir}
print("Conector GCS descargado correctamente.")

import time
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip

# 4. Configuración de Spark con soporte Delta y GCS
builder = (
    SparkSession.builder
    .appName("Forus_Fase2_Silver")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "8")
    # Indicadores para leer rutas gs://
    .config("spark.hadoop.fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
    .config("spark.hadoop.fs.AbstractFileSystem.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
    .config("spark.hadoop.google.cloud.auth.service.account.enable", "true")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)

# 5. Definición de Rutas en la Nube
NOMBRE_BUCKET = "data-lake-retail"
RUTA_BASE = f"gs://{NOMBRE_BUCKET}"

RUTA_RAW = f"{RUTA_BASE}/raw"
RUTA_BRONZE = f"{RUTA_BASE}/bronze"
RUTA_SILVER = f"{RUTA_BASE}/silver"
RUTA_GOLD = f"{RUTA_BASE}/gold"

print("Rutas configuradas correctamente apuntando a GCS.")

Autenticación con GCP exitosa.
Conector GCS descargado correctamente.
Spark version: 3.4.1
Rutas configuradas correctamente apuntando a GCS.


## 1. Lectura desde Bronze

In [2]:
# ==========================================
# 1. Lectura desde Capa Bronze
# ==========================================
# Apuntamos a la carpeta exacta que creamos en el notebook 01_Capa_Bronze
ruta_origen_bronze = f"{RUTA_BRONZE}/venta_tiendas_delta"

print(f"Conectando a GCS y leyendo tabla Delta desde: {ruta_origen_bronze}...")

# Spark lee directamente la tabla Delta desde Cloud Storage
df_bronze = spark.read.format("delta").load(ruta_origen_bronze)

print("Registros leídos desde Bronze:", df_bronze.count())
print("\n--- Esquema de entrada (Aún con tipos de datos incorrectos) ---")
df_bronze.printSchema()
df_bronze.show(5, truncate=False)

Conectando a GCS y leyendo tabla Delta desde: gs://data-lake-retail/bronze/venta_tiendas_delta...
Registros leídos desde Bronze: 2250970

--- Esquema de entrada (Aún con tipos de datos incorrectos) ---
root
 |-- id_canal: string (nullable = true)
 |-- numero_transaccion: string (nullable = true)
 |-- numero_pos: string (nullable = true)
 |-- numero_boleta: string (nullable = true)
 |-- fecha_transaccion: string (nullable = true)
 |-- cod_tienda_facturacion: string (nullable = true)
 |-- tipo_documento: string (nullable = true)
 |-- id_producto: string (nullable = true)
 |-- unidades: string (nullable = true)
 |-- venta: string (nullable = true)
 |-- costo: string (nullable = true)

+--------+------------------+----------+-------------+-------------------------+----------------------+--------------+-----------+--------+-----+-----+
|id_canal|numero_transaccion|numero_pos|numero_boleta|fecha_transaccion        |cod_tienda_facturacion|tipo_documento|id_producto|unidades|venta|costo|
+----

## 2. Transformaciones Silver\n\nReglas aplicadas:\n\n- Conversión de columnas numéricas.\n- Normalización de `fecha_transaccion`.\n- Creación de `fecha_venta` como tipo `date`.\n- Creación de `fecha_venta_texto` con guiones.\n- Cálculo de `margen` y `margen_porcentaje`.\n- Eliminación de registros inválidos críticos.

In [3]:
# ==========================================
# 2. Transformaciones Silver y Reglas de Calidad
# ==========================================
from pyspark.sql.functions import (
    col, regexp_replace, trim, to_timestamp, to_date, date_format,
    year, month, dayofmonth, when, lit
)
import time

inicio = time.time()
print("Aplicando reglas de transformación y limpieza...")

# 1. Limpieza del sufijo " CL" y parseo de fecha original
fecha_sin_zona = regexp_replace(col("fecha_transaccion"), " CL$", "")

# Aplicamos transformaciones a un DataFrame temporal para poder auditar
df_silver_temp = (
    df_bronze
    # Deduplicación inicial (Buena práctica)
    .dropDuplicates()

    # Casteo de tipos de datos
    .withColumn("id_canal", col("id_canal").cast("int"))
    .withColumn("numero_transaccion", col("numero_transaccion").cast("long"))
    .withColumn("numero_pos", col("numero_pos").cast("int"))
    .withColumn("numero_boleta", col("numero_boleta").cast("long"))
    .withColumn("cod_tienda_facturacion", col("cod_tienda_facturacion").cast("int"))
    .withColumn("id_producto", col("id_producto").cast("long"))
    .withColumn("unidades", col("unidades").cast("int"))
    .withColumn("venta", col("venta").cast("double"))
    .withColumn("costo", col("costo").cast("double"))
    .withColumn("tipo_documento", trim(col("tipo_documento")))

    # Transformación solicitada: slash a guion
    .withColumn("fecha_transaccion_guion", regexp_replace(col("fecha_transaccion"), "/", "-"))

    # Conversión profesional a timestamp y date
    .withColumn("fecha_timestamp", to_timestamp(fecha_sin_zona, "dd/MM/yyyy hh:mm:ss a"))
    .withColumn("fecha_venta", to_date(col("fecha_timestamp")))
    .withColumn("fecha_venta_texto", date_format(col("fecha_venta"), "yyyy-MM-dd"))

    # Variables derivadas comerciales
    .withColumn("anio", year(col("fecha_venta")))
    .withColumn("mes", month(col("fecha_venta")))
    .withColumn("dia", dayofmonth(col("fecha_venta")))
    .withColumn("margen", col("venta") - col("costo"))
    .withColumn(
        "margen_porcentaje",
        when(col("venta") > 0, (col("venta") - col("costo")) / col("venta")).otherwise(lit(None))
    )
)

# 2. Validación de calidad (ANTES de filtrar)
registros_fecha_nula = df_silver_temp.filter(col("fecha_venta").isNull()).count()
total_original = df_silver_temp.count()

# 3. Reglas mínimas de calidad (Filtros críticos)
df_silver = (
    df_silver_temp
    .filter(col("fecha_venta").isNotNull())
    .filter(col("id_producto").isNotNull())
    .filter(col("venta").isNotNull())
    .filter(col("unidades").isNotNull())
)

tiempo_transformacion = time.time() - inicio

print("\n--- RESUMEN DE TRANSFORMACIONES ---")
print(f"Registros originales en Bronze (sin duplicados): {total_original}")
print(f"Registros Silver finales (limpios): {df_silver.count()}")
print(f"Tiempo transformación Silver: {round(tiempo_transformacion, 2)} segundos")

if registros_fecha_nula > 0:
    print(f"⚠️ ADVERTENCIA: Se descartaron {registros_fecha_nula} registros porque sus fechas eran inválidas y no se pudieron transformar.")
else:
    print("✅ VALIDACIÓN EXITOSA: Todas las fechas fueron parseadas correctamente.")

print("\n--- ESQUEMA RESULTANTE (SILVER) ---")
df_silver.printSchema()

print("\n--- MUESTRA DE FECHAS Y MÁRGENES ---")
df_silver.select(
    "fecha_transaccion", "fecha_transaccion_guion", 
    "fecha_venta", "venta", "costo", "margen_porcentaje"
).show(10, truncate=False)

Aplicando reglas de transformación y limpieza...

--- RESUMEN DE TRANSFORMACIONES ---
Registros originales en Bronze (sin duplicados): 2249970
Registros Silver finales (limpios): 2249970
Tiempo transformación Silver: 51.51 segundos
✅ VALIDACIÓN EXITOSA: Todas las fechas fueron parseadas correctamente.

--- ESQUEMA RESULTANTE (SILVER) ---
root
 |-- id_canal: integer (nullable = true)
 |-- numero_transaccion: long (nullable = true)
 |-- numero_pos: integer (nullable = true)
 |-- numero_boleta: long (nullable = true)
 |-- fecha_transaccion: string (nullable = true)
 |-- cod_tienda_facturacion: integer (nullable = true)
 |-- tipo_documento: string (nullable = true)
 |-- id_producto: long (nullable = true)
 |-- unidades: integer (nullable = true)
 |-- venta: double (nullable = true)
 |-- costo: double (nullable = true)
 |-- fecha_transaccion_guion: string (nullable = true)
 |-- fecha_timestamp: timestamp (nullable = true)
 |-- fecha_venta: date (nullable = true)
 |-- fecha_venta_texto: stri

## 3. Validaciones de calidad

In [4]:
# ==========================================
# 3. Reporte de Calidad de Datos (QA Silver)
# ==========================================
from pyspark.sql.functions import sum as spark_sum, col, when

print("--- AUDITORÍA DE CALIDAD DE DATOS ---")
print("📌 Nota Comercial: Las ventas o unidades negativas representan devoluciones, no errores de sistema.\n")

validaciones = df_silver.select(
    # Errores críticos (deberían ser 0 gracias a nuestros filtros anteriores)
    spark_sum(when(col("fecha_venta").isNull(), 1).otherwise(0)).alias("fechas_invalidas"),
    spark_sum(when(col("id_producto").isNull(), 1).otherwise(0)).alias("productos_invalidos"),
    
    # Comportamiento de negocio (Devoluciones)
    spark_sum(when(col("venta") < 0, 1).otherwise(0)).alias("ventas_devolucion"),
    spark_sum(when(col("unidades") <= 0, 1).otherwise(0)).alias("unidades_cero_o_dev")
)

validaciones.show()

--- AUDITORÍA DE CALIDAD DE DATOS ---
📌 Nota Comercial: Las ventas o unidades negativas representan devoluciones, no errores de sistema.

+----------------+-------------------+-----------------+-------------------+
|fechas_invalidas|productos_invalidos|ventas_devolucion|unidades_cero_o_dev|
+----------------+-------------------+-----------------+-------------------+
|               0|                  0|           197119|             197008|
+----------------+-------------------+-----------------+-------------------+



## 4. Escritura Silver particionada\n\nSe particiona por `anio` y `mes` para mejorar filtros temporales y consultas analíticas.

In [5]:
# ==========================================
# 4. Escritura Silver Particionada (Delta Lake)
# ==========================================
import time

# Apuntamos a la carpeta silver de tu bucket
ruta_destino_silver = f"{RUTA_SILVER}/venta_tiendas_delta"
print(f"Guardando datos limpios y particionados en Silver: {ruta_destino_silver} ...")

inicio_escritura = time.time()

(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("anio", "mes") # 🔥 Optimización clave para consultas futuras
    .save(ruta_destino_silver)
)

tiempo_escritura = time.time() - inicio_escritura
print(f"✅ ¡Guardado exitoso en GCS! Tiempo: {round(tiempo_escritura, 2)} segundos")

Guardando datos limpios y particionados en Silver: gs://data-lake-retail/silver/venta_tiendas_delta ...
✅ ¡Guardado exitoso en GCS! Tiempo: 151.65 segundos


## 5. Linaje Silver

In [6]:
# ==========================================
# 5. Evidencia de Linaje Silver (Gobierno de Datos)
# ==========================================
linaje_silver = {
    "dataset": "venta_tiendas",
    "origen": ruta_origen_bronze,   # gs://data-lake-retail/bronze/venta_tiendas_delta
    "destino": ruta_destino_silver, # gs://data-lake-retail/silver/venta_tiendas_delta
    "capa": "Silver",
    "formato": "Delta Lake (Particionado)",
    "transformaciones": [
        "Casting de variables numéricas",
        "Parseo de fecha_transaccion eliminando sufijo 'CL'",
        "fecha_transaccion con slash transformada a fecha_transaccion_guion",
        "Creación de fecha_venta tipo date en formato yyyy-MM-dd",
        "Cálculo de margen y margen_porcentaje (manejando divisiones por cero)",
        "Filtros de calidad sobre fecha, producto, venta y unidades",
        "Particionamiento físico por anio y mes"
    ]
}

print("--- DOCUMENTACIÓN DE LINAJE SILVER ---")
for k, v in linaje_silver.items():
    if isinstance(v, list):
        print(f"{k.upper()}:")
        for item in v:
            print(f"  - {item}")
    else:
        print(f"{k.upper()}: {v}")

--- DOCUMENTACIÓN DE LINAJE SILVER ---
DATASET: venta_tiendas
ORIGEN: gs://data-lake-retail/bronze/venta_tiendas_delta
DESTINO: gs://data-lake-retail/silver/venta_tiendas_delta
CAPA: Silver
FORMATO: Delta Lake (Particionado)
TRANSFORMACIONES:
  - Casting de variables numéricas
  - Parseo de fecha_transaccion eliminando sufijo 'CL'
  - fecha_transaccion con slash transformada a fecha_transaccion_guion
  - Creación de fecha_venta tipo date en formato yyyy-MM-dd
  - Cálculo de margen y margen_porcentaje (manejando divisiones por cero)
  - Filtros de calidad sobre fecha, producto, venta y unidades
  - Particionamiento físico por anio y mes
